# Machine-learning–accelerated CFD

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/ml_accelerated_cfd)

A plain finite-volume solver on a grid 16 times coarser than the truth smears the shocks of the viscous Burgers equation. A tiny CNN learns a correction to the face fluxes, trained by backpropagating through an unrolled coarse simulation. On initial conditions it has never seen, the learned solver follows the fine solution several times more closely, at the same coarse resolution, while conserving $\int u\,dx$ exactly.

**Vector form**

$$\partial_t u + \partial_x\big(\tfrac12 u^2 - \nu\,\partial_x u\big) = 0,\qquad \mathcal L(\theta) = \Big\langle \textstyle\sum_{s=1}^{S}\big\lVert u_\theta^{(s)} - \bar u_\text{fine}^{(s)}\big\rVert^2\Big\rangle$$

**Index (tensor) form**

$$\dot u_{bi} = -\frac{F_{b,i+\frac12} - F_{b,i-\frac12}}{\Delta x},\qquad F_{b,i+\frac12} = F^\text{Rus}_{b,i+\frac12} - \nu\frac{u_{b,i+1} - u_{bi}}{\Delta x} + \delta F_{b,i+\frac12}[u],\qquad h^{(1)}_{bci} = \sigma\big(K^{(1)}_{ck}\,u_{b,i+k} + \beta_c\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{bi}$ | `u` | (B, N_c) | cell average in coarse cell $i$ for trajectory $b$ (periodic) |
| $F^\text{Rus}_{i+\frac12}$ | `rusanov_flux(u)` | (B, N_c) | $\tfrac14(u_i^2 + u_{i+1}^2) - \tfrac12\alpha(u_{i+1} - u_i)$, $\alpha = \max(\lvert u_i\rvert, \lvert u_{i+1}\rvert)$; $u_{i+1}$ is roll(u, -1) |
| $F_{i+\frac12} - F_{i-\frac12}$ | `F - roll(F, 1, dims=-1)` | (B, N_c) | what flows out of one cell flows into the next ⇒ $\sum_i u_i$ conserved |
| $K^{(1)}_{ck}\,u_{b,i+k}$ | `Conv1d(1, 16, 5, padding_mode='circular')` | (B, 16, N_c) | repeated $k$ ⇒ summed: a 5-cell stencil, periodic padding |
| $\delta F_{i+\frac12}[u]$ | `FluxCorrection()(u)` | (B, N_c) | learned flux correction from three convolutions (zero at the start) |
| $\bar u_{\text{fine},I}$ | `u_fine.unflatten(-1, (-1, f)).mean(-1)` | (T, B, N_c) | fine solution averaged over the $f = N_f/N_c$ fine cells in each coarse cell |
| $\nabla_\theta\mathcal L$ | `loss.backward()` | — | autograd through all $S$ unrolled time steps of the solver |

Units: the domain is $[0, 2\pi)$, periodic. The truth is the same finite-volume scheme on 512 cells, with Heun time stepping sub-cycled to stay stable. Training uses 32 random smooth waves, and the test uses 8 others from a different seed, run 50% longer ($t = 3$) than the training data. At each Adam step the learned solver starts from the true coarse state at random times, runs $S$ steps, and is penalised for drifting from the averaged truth. Gradients flow back through every time step, so the network learns corrections that stay good when they are fed their own output. Two design choices keep it honest: $\delta F$ enters as a flux, so $\sum_i u_i$ is conserved to round-off, and the last layer starts at zero, so training begins from the plain solver. The idea follows Bar-Sinai, Hoyer, Hickey & Brenner (PNAS, 2019), who learned data-driven discretisations of exactly this kind of equation, and Kochkov, Smith, Alieva, Wang, Brenner & Hoyer (PNAS, 2021). They learned interpolation coefficients inside a 2D Navier–Stokes solver and reported 8–10× coarser grids at the same accuracy, roughly 40–80× faster. This toy shows the mechanism, not those speed-ups. A learned correction can fail on inputs far from its training data. Try training for fewer steps, a shorter unroll, or a viscosity the network was not trained on.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/ml_accelerated_cfd/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import mean

N_FINE = 512  # ground-truth resolution
N_TRAIN, N_TEST = 32, 8  # training trajectories / held-out trajectories (different random seed)
T_TRAIN, T_TEST = 100, 150  # coarse steps: trained on t ≤ 2, tested out to t = 3


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    torch.manual_seed(0)  # the network's initial weights
    nu, n_coarse = params["nu"], int(params["n_coarse"])
    n_iter = 10 if fast_mode() else params["train_iters"]
    g = solve.setup(N_FINE, n_coarse, dev)
    print(f"fine grid {N_FINE} cells, coarse grid {n_coarse} cells ({g['factor']}× coarser), Δt = {solve.DT_COARSE}")

    # ground truth: fine solver, averaged onto the coarse cells
    _, truth = solve.ground_truth(
        tensors.random_initial(g["x_f"], N_TRAIN, seed=1), nu, g["dx_f"], g["factor"], T_TRAIN
    )
    fine_test, truth_test = solve.ground_truth(
        tensors.random_initial(g["x_f"], N_TEST, seed=2), nu, g["dx_f"], g["factor"], T_TEST
    )

    # learn the flux correction by differentiating through `unroll` coarse steps
    model = tensors.FluxCorrection().to(dev)
    n_weights = sum(p.numel() for p in model.parameters())
    losses = solve.train(model, truth, nu, g["dx_c"], n_iter, params["unroll"])
    print(f"CNN with {n_weights} weights, {n_iter} Adam steps: loss {losses[0]:.2e} → {losses[-1]:.2e}")

    # held-out initial conditions, run 50% longer than anything seen in training
    with torch.no_grad():
        plain = solve.rollout(truth_test[0], nu, g["dx_c"], T_TEST)  # (T, B, N_c)
        learned = solve.rollout(truth_test[0], nu, g["dx_c"], T_TEST, model)
    err_plain = solve.rms_error(plain, truth_test)  # (T, B)
    err_learned = solve.rms_error(learned, truth_test)
    for name, e in [("plain coarse", err_plain), ("learned coarse", err_learned)]:
        print(f"{name:>15}: RMS error vs fine (held-out mean): t = 1: {e[50].mean():.4f}, t = 3: {e[-1].mean():.4f}")
    better = (err_learned.mean(0) < err_plain.mean(0)).sum().item()
    print(f"learned beats plain on {better}/{N_TEST} unseen initial conditions")
    drift = (mean(learned) - mean(learned[0])).abs().max().item()
    print(f"mean ⟨u⟩ drift of the learned solver: {drift:.1e} (conserved: δF is a flux)")

    times = torch.arange(T_TEST + 1) * solve.DT_COARSE
    return plot.animate(
        g["x_f"], g["x_c"], times, fine_test, plain, learned, err_plain, err_learned, losses,
        out_dir / "ml_accelerated_cfd.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/ml_accelerated_cfd/equation.py
r"""Physics → tensor translation.

Viscous Burgers in conservation form, u_t + ∂_x F(u) = 0 with flux F = ½u² - ν u_x, on a periodic
domain. A finite-volume scheme stores cell averages u_i and updates them with the flux through each face:

    du_i/dt = -(F_{i+½} - F_{i-½}) / Δx

Whatever F_{i+½} is, the same number leaves cell i and enters cell i+1, so Σ_i u_i Δx is conserved exactly.
The plain scheme uses the Rusanov (local Lax–Friedrichs) flux plus a centred viscous flux:

    F_{i+½} = ½(½u_i² + ½u_{i+1}²) - ½ α_{i+½} (u_{i+1} - u_i) - ν (u_{i+1} - u_i)/Δx,   α = max(|u_i|, |u_{i+1}|)

The learned scheme adds a correction from a small convolutional network (in the spirit of Kochkov et al.,
PNAS 2021, who learned interpolation stencils for Navier–Stokes):

    F_{i+½} → F_{i+½} + δF_{i+½}[u],      δF = CNN(u):   h^{(1)}_{c,i} = σ(K^{(1)}_{c,k} u_{i+k} + β_c),   …

Each convolution sums over the input channel and the stencil offset k (the circular padding is the periodic
boundary). Because δF is a flux, the learned scheme conserves Σ_i u_i as exactly as the plain one does.
"""

import torch


def rusanov_flux(u: torch.Tensor) -> torch.Tensor:
    r"""F^{adv}_{i+½} = ¼(u_i² + u_{i+1}²) - ½ max(|u_i|, |u_{i+1}|)(u_{i+1} - u_i).

    u: (B, N) → (B, N); entry i is the flux through face i+½."""
    right = torch.roll(u, -1, dims=-1)  # u_{i+1} (periodic)
    alpha = torch.maximum(u.abs(), right.abs())
    return 0.25 * (u**2 + right**2) - 0.5 * alpha * (right - u)


def viscous_flux(u: torch.Tensor, nu: float, dx: float) -> torch.Tensor:
    r"""F^{visc}_{i+½} = -ν (u_{i+1} - u_i) / Δx.   (B, N) → (B, N)"""
    return -nu * (torch.roll(u, -1, dims=-1) - u) / dx


def rhs(u: torch.Tensor, nu: float, dx: float, correction=None) -> torch.Tensor:
    r"""du_i/dt = -(F_{i+½} - F_{i-½})/Δx with F = Rusanov + viscous (+ learned δF).   (B, N) → (B, N)"""
    F = rusanov_flux(u) + viscous_flux(u, nu, dx)
    if correction is not None:
        F = F + correction(u)
    return -(F - torch.roll(F, 1, dims=-1)) / dx


def step(u: torch.Tensor, dt: float, nu: float, dx: float, correction=None) -> torch.Tensor:
    """One Heun (2nd-order Runge–Kutta) step; with a correction it stays differentiable for training."""
    k1 = rhs(u, nu, dx, correction)
    k2 = rhs(u + dt * k1, nu, dx, correction)
    return u + 0.5 * dt * (k1 + k2)


def cell_average(u_fine: torch.Tensor, factor: int) -> torch.Tensor:
    r"""ū_I = (1/f) Σ_{i ∈ I} u_i: fine cells → coarse cells (f fine cells each).   (..., N_f) → (..., N_f / f)"""
    return u_fine.unflatten(-1, (-1, factor)).mean(-1)


def mean(u: torch.Tensor) -> torch.Tensor:
    r"""⟨u⟩ = (1/N) Σ_i u_i — conserved on a periodic domain.   (..., N) → (...)"""
    return u.mean(-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/ml_accelerated_cfd/tensors.py
"""Build the concrete tensors: the grids, random initial conditions and the tiny flux-correction network."""

import math

import torch
from torch import nn

LENGTH = 2 * math.pi  # periodic domain [0, 2π)


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """Cell centres x_i on [0, 2π) and the cell width Δx."""
    dx = LENGTH / n
    return (torch.arange(n, device=device, dtype=torch.float32) + 0.5) * dx, dx


def random_initial(x: torch.Tensor, n: int, seed: int, n_modes: int = 4) -> torch.Tensor:
    """n random smooth waves u_0(x) = c + Σ_m a_m sin(m x + φ_m), m = 1…n_modes: shape (n, N).

    Amplitudes fall off as 1/m and are rescaled so max|u_0 - c| = 1; c is a small random mean. Fixed seed.
    """
    g = torch.Generator().manual_seed(seed)
    m = torch.arange(1, n_modes + 1, dtype=torch.float32)  # (M,)
    a = (2 * torch.rand(n, n_modes, generator=g) - 1) / m  # (n, M)
    phase = 2 * math.pi * torch.rand(n, n_modes, generator=g)
    c = 0.4 * (torch.rand(n, 1, generator=g) - 0.5)
    waves = torch.einsum("bm,bmx->bx", a, torch.sin(m[None, :, None] * x.cpu()[None, None, :] + phase[..., None]))
    waves = waves / waves.abs().amax(dim=1, keepdim=True)
    return (c + waves).to(x.device)


class FluxCorrection(nn.Module):
    """δF_{i+½} = CNN(u): three periodic 1D convolutions (stencil 5 cells), a few hundred weights.

    The last layer starts at zero, so before training the learned solver *is* the plain solver.
    """

    def __init__(self, channels: int = 16, kernel: int = 5):
        super().__init__()
        conv = dict(kernel_size=kernel, padding=kernel // 2, padding_mode="circular")
        self.net = nn.Sequential(
            nn.Conv1d(1, channels, **conv),
            nn.GELU(),
            nn.Conv1d(channels, channels, **conv),
            nn.GELU(),
            nn.Conv1d(channels, 1, **conv),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        """u: (B, N) → δF: (B, N), entry i is the correction at face i+½."""
        return self.net(u[:, None, :])[:, 0, :]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/ml_accelerated_cfd/solve.py
"""Fine ground truth, plain coarse solver, and training the flux correction through an unrolled simulation."""

import math

import torch

from problems._common.progress import steps

from . import tensors
from .equation import cell_average, step

DT_COARSE = 0.02  # coarse time step (CFL ≈ 0.2 on 32 cells)


def substeps(dx_fine: float, nu: float, u_max: float = 2.0) -> int:
    """Fine steps per coarse step, so the fine solver respects both the CFL and the diffusion limit."""
    dt_max = min(0.3 * dx_fine / u_max, 0.2 * dx_fine**2 / nu)
    return math.ceil(DT_COARSE / dt_max)


@torch.no_grad()
def ground_truth(
    u0_fine: torch.Tensor, nu: float, dx_fine: float, factor: int, n_steps: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Run the fine solver; return its snapshots every coarse step (n_steps + 1, B, N_f) and their
    coarse-cell averages (n_steps + 1, B, N_c), which are what a perfect coarse solver would produce."""
    k = substeps(dx_fine, nu)
    u, out = u0_fine, [u0_fine]
    for _ in steps(n_steps, "fine solver"):
        for _ in range(k):
            u = step(u, DT_COARSE / k, nu, dx_fine)
        out.append(u)
    fine = torch.stack(out)
    return fine, cell_average(fine, factor)


def rollout(u0: torch.Tensor, nu: float, dx: float, n_steps: int, correction=None) -> torch.Tensor:
    """Coarse trajectory (n_steps + 1, B, N_c), plain (correction=None) or learned."""
    us = [u0]
    for _ in range(n_steps):
        us.append(step(us[-1], DT_COARSE, nu, dx, correction))
    return torch.stack(us)


def train(
    model: torch.nn.Module, truth: torch.Tensor, nu: float, dx: float, n_iter: int, unroll: int, seed: int = 0
) -> list[float]:
    """Fit δF by backpropagating through `unroll` coarse steps (autograd through the whole simulation).

    Each iteration picks random start times and trajectories from truth (T, B, N_c), runs the learned
    solver from those states and compares every step with the truth: L = mean (u_learned - u_true)².
    """
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(n_iter, 1))
    T, B, _ = truth.shape
    losses = []
    for _ in steps(n_iter, "training"):
        t0 = torch.randint(0, T - unroll, (32,), generator=g)
        b = torch.randint(0, B, (32,), generator=g)
        target = torch.stack([truth[t0 + s, b] for s in range(unroll + 1)])  # (unroll+1, 32, N_c)
        pred = rollout(target[0], nu, dx, unroll, model)
        loss = ((pred[1:] - target[1:]) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
    return losses


def rms_error(u: torch.Tensor, truth: torch.Tensor) -> torch.Tensor:
    """Root-mean-square error over space at each time and trajectory: (T, B, N) → (T, B)."""
    return ((u - truth) ** 2).mean(-1).sqrt()


def setup(n_fine: int, n_coarse: int, device: torch.device) -> dict:
    """Both grids and the coarsening factor."""
    x_f, dx_f = tensors.grid(n_fine, device)
    x_c, dx_c = tensors.grid(n_coarse, device)
    return {"x_f": x_f, "dx_f": dx_f, "x_c": x_c, "dx_c": dx_c, "factor": n_fine // n_coarse}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/ml_accelerated_cfd/plot.py
"""Top: two unseen initial conditions, fine vs plain coarse vs learned coarse.
Bottom: error vs time and the training loss."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator

from problems._common.plotting import frame_indices, save_gif

FINE, PLAIN, LEARNED = "#111827", "#ea580c", "#2563eb"


def animate(
    x_f: torch.Tensor, x_c: torch.Tensor, times: torch.Tensor, fine: torch.Tensor, plain: torch.Tensor,
    learned: torch.Tensor, err_plain: torch.Tensor, err_learned: torch.Tensor, losses: list, path: Path,
) -> Path:  # fmt: skip
    xf, xc, t = x_f.cpu().numpy(), x_c.cpu().numpy(), times.cpu().numpy()
    uf, up, ul = fine.cpu().numpy(), plain.cpu().numpy(), learned.cpu().numpy()
    ep, el = err_plain.mean(1).cpu().numpy(), err_learned.mean(1).cpu().numpy()
    edges = np.append(xc - 0.5 * (xc[1] - xc[0]), xc[-1] + 0.5 * (xc[1] - xc[0]))  # coarse cell edges

    fig = plt.figure(figsize=(7.4, 4.8))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
    curves, titles = [], []
    for b in range(2):  # two held-out runs
        ax = fig.add_subplot(grid[0, b])
        f = ax.plot(xf, uf[0, b], color=FINE, lw=1.2, label=f"fine ({len(xf)} cells)")[0]
        p = ax.stairs(up[0, b], edges, color=PLAIN, lw=1.5, label=f"plain ({len(xc)} cells)")
        q = ax.stairs(ul[0, b], edges, color=LEARNED, lw=1.5, label=f"learned ({len(xc)} cells)")
        lo, hi = uf[:, b].min(), uf[:, b].max()
        ax.set_ylim(lo - 0.15 * (hi - lo), hi + 0.3 * (hi - lo))
        ax.set_xlim(0, edges[-1])
        ax.set_xlabel("x")
        ax.set_ylabel("u")
        if b == 0:
            ax.legend(fontsize=7, loc="upper left", ncol=1)
        titles.append(ax.set_title(" ", fontsize=9))
        curves.append((f, p, q, b))

    ax = fig.add_subplot(grid[1, 0])
    ax.plot(t, ep, color=PLAIN, lw=1.8, label="plain")
    ax.plot(t, el, color=LEARNED, lw=1.8, label="learned")
    ax.axvspan(2.0, t[-1], color="#f3f4f6", lw=0)
    ax.text(2.05, 0.95 * max(ep.max(), el.max()), "beyond training", fontsize=7, color="#6b7280", va="top")
    ax.set_xlim(0, t[-1])
    ax.set_ylim(0, 1.1 * max(ep.max(), el.max()))
    ax.set_xlabel("time t")
    ax.set_ylabel("RMS error (unseen runs)")
    ax.legend(fontsize=7, loc="upper left")
    cursor = ax.axvline(0, color="#f59e0b", lw=1.2)

    ax = fig.add_subplot(grid[1, 1])
    ax.semilogy(np.arange(1, len(losses) + 1), losses, color="#374151", lw=1.2)
    ax.yaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0e}"))  # plain text: mathtext redraws slowly
    ax.set_xlabel("Adam step")
    ax.set_ylabel("unrolled training loss")

    def update(step: int):
        for f, p, q, b in curves:
            f.set_ydata(uf[step, b])
            p.set_data(up[step, b])
            q.set_data(ul[step, b])
            titles[b].set_text(f"unseen run {b + 1},  t = {t[step]:.2f}")
        cursor.set_xdata([t[step]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 60), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'nu': 0.02,
    'n_coarse': '32',
    'train_iters': 150,
    'unroll': 12
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.ml_accelerated_cfd")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.ml_accelerated_cfd.main")
problem_dir = Path("problems/fluids/ml_accelerated_cfd")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
